<h4>Self Attention is a mechanism/technique which we use to convert our static embedding vectors of our tokens to a contextual vector/embedding with an arbitrary provided dimension</h4>

```self-attention = softmax((Q.transpose(K))/root(d_k)).V```

In [25]:
sentence = " Honesty is the best policy, so be honest in life"

dc = {s: i for i,s in enumerate(sorted(sentence.replace(',','').split()))}

print(dc)

{'Honesty': 0, 'be': 1, 'best': 2, 'honest': 3, 'in': 4, 'is': 5, 'life': 6, 'policy': 7, 'so': 8, 'the': 9}


In [26]:
import torch
import torch.nn as nn

In [27]:
sentence_int = torch.tensor([dc[s] for s in sentence.replace(',','').split()])
print(sentence_int)

tensor([0, 5, 9, 2, 7, 8, 1, 3, 4, 6])


In [28]:
torch.manual_seed(42)
embed = nn.Embedding(10,16)
embedded_sentence = embed(sentence_int).detach()

print(embedded_sentence)
print(embedded_sentence.shape)

tensor([[ 1.9269e+00,  1.4873e+00,  9.0072e-01, -2.1055e+00,  6.7842e-01,
         -1.2345e+00, -4.3067e-02, -1.6047e+00, -7.5214e-01,  1.6487e+00,
         -3.9248e-01, -1.4036e+00, -7.2788e-01, -5.5943e-01, -7.6884e-01,
          7.6245e-01],
        [ 1.0868e-02, -3.3874e-01, -1.3407e+00, -5.8537e-01,  5.3619e-01,
          5.2462e-01,  1.1412e+00,  5.1644e-02,  7.4395e-01, -4.8158e-01,
         -1.0495e+00,  6.0390e-01, -1.7223e+00, -8.2777e-01,  1.3347e+00,
          4.8354e-01],
        [-9.7267e-01,  9.5846e-01,  1.6192e+00,  1.4506e+00,  2.6948e-01,
         -2.1038e-01, -7.3280e-01,  1.0430e-01,  3.4875e-01,  9.6759e-01,
         -4.6569e-01,  1.6048e+00, -2.4801e+00, -4.1754e-01, -1.1955e+00,
          8.1234e-01],
        [-1.3847e+00, -8.7124e-01, -2.2337e-01,  1.7174e+00,  3.1888e-01,
         -4.2452e-01,  3.0572e-01, -7.7459e-01, -1.5576e+00,  9.9564e-01,
         -8.7979e-01, -6.0114e-01, -1.2742e+00,  2.1228e+00, -1.2347e+00,
         -4.8791e-01],
        [-1.4570e+00

In [29]:
torch.manual_seed(42)

d = embedded_sentence.shape[1]

d_q, d_k, d_v = 24,24,28

W_query = torch.nn.Parameter(torch.rand(d_q,d))
W_key = torch.nn.Parameter(torch.rand(d_k,d))
W_value = torch.nn.Parameter(torch.rand(d_v,d))


In [30]:
x_4 = embedded_sentence[3]
query_4 = W_query.matmul(x_4)
key_4 = W_key.matmul(x_4)
value_4 = W_value.matmul(x_4)

print(query_4.shape)
print(key_4.shape)
print(value_4.shape)
 

torch.Size([24])
torch.Size([24])
torch.Size([28])


In [31]:
keys = W_key.matmul(embedded_sentence.T).T
values = W_value.matmul(embedded_sentence.T).T

print(f"Shape of key matrix is {keys.shape}")
print(f"Shape of value matrix is {values.shape}")

Shape of key matrix is torch.Size([10, 24])
Shape of value matrix is torch.Size([10, 28])


In [32]:
omega_44 = query_4.dot(keys[4])
print(omega_44)

tensor(-87.9399, grad_fn=<DotBackward0>)


In [33]:
omega_4 = query_4.matmul(keys.T)
print(omega_4)

tensor([  40.8972,   31.6913,  -41.4021,  131.2128,  -87.9399, -123.2757,
        -281.5240,   36.4287,  -97.1688,  -74.0868],
       grad_fn=<SqueezeBackward4>)


In [57]:
import torch.nn.functional as F

attention_weights_4 = F.softmax(omega_4 / d_k ** 0.5, dim=0)
print(attention_weights_4)

tensor([9.8521e-09, 1.5046e-09, 4.9854e-16, 1.0000e+00, 3.7335e-20, 2.7517e-23,
        2.5757e-37, 3.9573e-09, 5.6752e-21, 6.3124e-19],
       grad_fn=<SoftmaxBackward0>)


In [58]:
context_embedding_4 = attention_weights_4.matmul(values)

In [59]:
print(context_embedding_4.shape)
print(context_embedding_4)

torch.Size([28])
tensor([-1.9569, -2.2661, -2.4877, -3.1000, -0.9784, -6.0997, -2.8222, -1.5047,
        -1.7224, -3.2717, -1.0175, -3.1253, -3.4689, -1.0962, -2.1339, -1.5737,
        -3.9808, -3.3262, -2.3460, -1.2702,  0.7139, -0.4141, -3.3865, -2.1503,
        -1.2775, -1.4216, -3.0825, -0.4842], grad_fn=<SqueezeBackward4>)


<h3>Multi head attention is a combination of multiple heads of self-attention</h3>

In [37]:
h = 3

multihead_W_query = torch.nn.Parameter(torch.rand(h,d_q,d))
multihead_W_key = torch.nn.Parameter(torch.rand(h,d_k,d))
multihead_W_value = torch.nn.Parameter(torch.rand(h,d_v,d))

In [40]:
multihead_query_4 = multihead_W_query.matmul(x_4)
print(multihead_query_4.shape)

torch.Size([3, 24])


In [41]:
multihead_key_4 = multihead_W_key.matmul(x_4)
multihead_value_4 = multihead_W_value.matmul(x_4)

In [43]:
stacked_inputs = embedded_sentence.T.repeat(3,1,1)
print(stacked_inputs.shape)

torch.Size([3, 16, 10])


In [45]:
multihead_keys = torch.bmm(multihead_W_key, stacked_inputs)
multihead_values = torch.bmm(multihead_W_value, stacked_inputs)

print(f"Multihead keys.shape -> {multihead_keys.shape}")
print(f"Multihead values.shape -> {multihead_values.shape}")

Multihead keys.shape -> torch.Size([3, 24, 10])
Multihead values.shape -> torch.Size([3, 28, 10])


In [48]:
multihead_keys = multihead_keys.permute(0,2,1)
multihead_values = multihead_values.permute(0,2,1)

print(f"Multihead keys.shape -> {multihead_keys.shape}")
print(f"Multihead values.shape -> {multihead_values.shape}")

Multihead keys.shape -> torch.Size([3, 10, 24])
Multihead values.shape -> torch.Size([3, 10, 28])


In [53]:
mutlihead_query_4 = multihead_query_4.unsqueeze(1)
mutlihead_query_4.shape

torch.Size([3, 1, 24])

In [56]:
multihead_omega_4 = torch.bmm(mutlihead_query_4, multihead_keys.permute(0,2,1))
print(multihead_omega_4.shape)

torch.Size([3, 1, 10])


In [63]:
multihead_attention_weights = F.softmax(multihead_omega_4 / d_k ** 0.5, dim=-1)
print(multihead_attention_weights.shape)

torch.Size([3, 1, 10])


In [64]:
context_vectors = torch.bmm(multihead_attention_weights, multihead_values)

In [65]:
context_vectors.shape

torch.Size([3, 1, 28])

In [66]:
print(context_vectors)

tensor([[[-2.3238, -2.3688, -1.6333, -4.5426,  0.9878, -2.1332, -1.1472,
          -1.6568, -4.0356, -3.2132, -2.4959, -2.6972, -2.7363, -4.3699,
          -1.6716, -0.4476, -4.6628, -4.0665, -2.2423, -1.5809, -1.5601,
          -0.9806, -3.7485, -2.5964, -3.1362, -1.1947, -4.0525, -0.8454]],

        [[-2.9318, -3.4798, -4.0832, -0.8530, -1.9446, -2.1007, -3.0893,
           0.0247, -2.0316, -2.7751, -3.2231, -1.2982, -3.8823, -1.7784,
          -0.6279, -4.2329, -1.7082, -1.9278, -0.9215, -3.1918, -1.1036,
          -0.1238, -0.2899,  0.4532, -3.2017, -2.5299, -2.7141, -0.5394]],

        [[-2.2775, -1.8641, -0.0320, -1.4851,  0.2902,  0.6466, -3.2740,
          -1.4119, -2.4178, -3.9945, -3.0832, -1.5822, -3.9163, -2.1161,
          -2.7802, -2.9929, -3.2966, -3.3661, -0.0171, -2.8743, -1.5765,
          -3.4208, -2.3863, -1.1230, -1.2888, -2.2759, -2.8346, -0.6555]]],
       grad_fn=<BmmBackward0>)
